In [402]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [403]:
import os
os.chdir('C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
import json
import warnings
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import config
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.inspection import permutation_importance
from src.utils import (
    read_csv_file, 
    save_csv_file, 
    agnostic_bacc_scorer,
    compute_bacc,
    compute_logloss,
    compute_recall_per_class,
    reduce_mem_usage
)
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from category_encoders import TargetEncoder, CountEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import xgboost as xgb
import catboost as cb
import lightgbm as lgb
import mlflow
from tqdm.notebook import tqdm
from pytabkit import (
	RealMLP_TD_Classifier,
	TabM_D_Classifier,
	Resnet_RTDL_D_Classifier,
	CatBoost_TD_Classifier,
	LGBM_TD_Classifier,
	XGB_TD_Classifier
)
from src.fe import *

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

### Prepare data

In [404]:
train_raw = read_csv_file(r'data\raw\train.csv')
test_raw = read_csv_file(r'data\raw\test.csv')

# Train and test ids
train_ids = train_raw['id']
test_ids = test_raw['id']

n_classes = 3

Reading file from: data\raw\train.csv
Reading file from: data\raw\test.csv


In [405]:
# X and y split
X = train_raw.drop(['id', 'irrigation_need'], axis=1)
y = train_raw['irrigation_need'].map(config.TARGET_MAP)

# Test data
test_data = test_raw.drop('id', axis=1)

In [406]:
# Reduce memory size of data
X = reduce_mem_usage(X)
test_data = reduce_mem_usage(test_data)

Memory before: 354.06 MB
Memory after: 327.63 MB
Reduced by: 26.44 MB (7.5%)
Memory before: 151.74 MB
Memory after: 140.41 MB
Reduced by: 11.33 MB (7.5%)


In [407]:
RUN = 'v3'

### Logistic Regression

In [203]:
drop_for_linear = ["water_source", "crop_type", "irrigation_type", "soil_type", "season", "region"]

X = X.drop(drop_for_linear, axis=1)
test_data = test_data.drop(drop_for_linear, axis=1)

# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [204]:
EXP_NAME = "LOGISTIC"
RUN_NAME = f"Logreg_seed{config.SEED}_{RUN}"

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # -------- STATIC LOGS --------
    mlflow.log_text("\n".join(X.columns.tolist()), "artifacts/all_feature_names.txt")
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")

    dtype_map = {col: str(dtype) for col, dtype in X.dtypes.items()}
    mlflow.log_dict(dtype_map, "artifacts/feature_dtypes.json")

    skf = StratifiedKFold(
        n_splits=config.N_FOLDS,
        shuffle=True,
        random_state=config.SEED
    )

    classes = np.unique(y)
    rank_cols = [f"{col}_rank" for col in num_cols]
    all_num_cols = num_cols + rank_cols

    final_oof_preds = np.zeros((len(X), n_classes))
    final_test_preds = np.zeros((len(test_data), n_classes))
    feature_importance = []

    bacc_scores = []
    ll_scores = []

    logged_flag = False

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):

        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        X_test  = test_data.copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        # -------- RANK FEATURES --------
        for col in num_cols:
            train_vals = X_train[col].values
            sorted_train = np.sort(train_vals)

            X_train[f"{col}_rank"] = rankdata(train_vals, method="average")
            X_valid[f"{col}_rank"] = np.searchsorted(sorted_train, X_valid[col].values)
            X_test[f"{col}_rank"]  = np.searchsorted(sorted_train, X_test[col].values)

        # -------- PIPELINE --------
        preprocess = ColumnTransformer([
            ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), cat_cols),
            ("num", StandardScaler(), all_num_cols),
        ], verbose_feature_names_out=False).set_output(transform="pandas")

        model = Pipeline([
            ("prep", preprocess),
            ("clf", LogisticRegression(**config.LOGISTIC_PARAMS))
        ])

        model.fit(X_train, y_train)

        # -------- LOG ONCE --------
        if not logged_flag:
            X_train_proc = model.named_steps["prep"].transform(X_train)

            mlflow.log_params({
                "seed": config.SEED,
                "n_folds": config.N_FOLDS,
                "n_rows": len(X),
                "n_cols_original": X.shape[1],
                "n_cols_processed": X_train_proc.shape[1],
                "processed_to_original_ratio": X_train_proc.shape[1] / X.shape[1],
                **config.LOGISTIC_PARAMS
            })

            mlflow.log_text(
                "\n".join(X_train_proc.columns.astype(str).tolist()),
                "artifacts/processed_feature_names.txt"
            )

            logged_flag = True

        # -------- FOLD RUN --------
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):

            mlflow.sklearn.log_model(model, name="model")

            # predictions
            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict_proba(X_test)
            final_test_preds += test_preds / config.N_FOLDS

            # feature importance
            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=3,
                n_jobs=-1,
                random_state=config.SEED
            )

            feature_importance.append(fi.importances_mean)

            del model, oof_preds, test_preds, fi, X_train, X_test
            gc.collect()

    # -------- FINAL METRICS --------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)

    metrics = {
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": np.std(bacc_scores),
        "std_logloss": np.std(ll_scores),
    }

    recall_per_class = compute_recall_per_class(y, final_oof_preds)
    for cls, rec in zip(classes, recall_per_class):
        metrics[f"recall_class_{cls}"] = rec

    mlflow.log_metrics(metrics)

    # -------- FEATURE IMPORTANCE --------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)

    fi_df = pd.DataFrame({
        "feature": X_valid.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")

    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)

# -------- SAVE FILES --------

oof_df = pd.DataFrame(final_oof_preds, columns=[str(c) for c in classes])
oof_df.insert(0, "id", train_ids)

save_csv_file(
    oof_df,
    os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv")
)

test_df = pd.DataFrame(final_test_preds, columns=[str(c) for c in classes])
test_df.insert(0, "id", test_ids)

save_csv_file(
    test_df,
    os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
)

# -------- OOF Residuals --------
y_onehot = np.eye(n_classes)[y]

oof_residuals = y_onehot - final_oof_preds

residual_df = pd.DataFrame(
    oof_residuals,
    columns=[f"residual_{c}" for c in classes]
)

residual_df.insert(0, "id", train_ids)

save_csv_file(
    residual_df,
    os.path.join(config.RESIDUAL_DIR, f"{RUN_NAME}_oof_residuals.csv")
)

KeyboardInterrupt: 

### SGD

In [222]:
X, _ = create_polynomial_features(X, config.BASE_NUM_COLS)
test_data, _ = create_polynomial_features(test_data, config.BASE_NUM_COLS)

Added 66 polynomial features
Added 66 polynomial features


In [223]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [224]:
EXP_NAME = "SGD"
RUN_NAME = f"SGD_seed{config.SEED}_{RUN}"

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # -------- STATIC LOGS --------
    mlflow.log_text("\n".join(X.columns.tolist()), "artifacts/all_feature_names.txt")
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")

    dtype_map = {col: str(dtype) for col, dtype in X.dtypes.items()}
    mlflow.log_dict(dtype_map, "artifacts/feature_dtypes.json")

    skf = StratifiedKFold(
        n_splits=config.N_FOLDS,
        shuffle=True,
        random_state=config.SEED
    )

    classes = np.unique(y)

    final_oof_preds = np.zeros((len(X), n_classes))
    final_test_preds = np.zeros((len(test_data), n_classes))
    feature_importance = []

    bacc_scores = []
    ll_scores = []

    logged_flag = False

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):

        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        X_test = test_data.copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]
        
        te = TargetEncoder(cols=cat_cols)

        X_train = te.fit_transform(X_train, y_train)
        X_valid = te.transform(X_valid)
        X_test = te.transform(X_test)

        # -------- PIPELINE --------
        preprocess = ColumnTransformer([
            #("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), cat_cols),
            ("num", StandardScaler(), num_cols + cat_cols),
        ], verbose_feature_names_out=False).set_output(transform="pandas")

        model = Pipeline([
            ("prep", preprocess),
            ("clf", SGDClassifier(**config.SGD_PARAMS))
        ])

        model.fit(X_train, y_train)

        # -------- LOG ONCE --------
        if not logged_flag:
            X_train_proc = model.named_steps["prep"].transform(X_train)

            mlflow.log_params({
                "seed": config.SEED,
                "n_folds": config.N_FOLDS,
                "n_rows": len(X),
                "n_cols_original": X.shape[1],
                "n_cols_processed": X_train_proc.shape[1],
                **config.SGD_PARAMS
            })

            mlflow.log_text(
                "\n".join(X_train_proc.columns.astype(str).tolist()),
                "artifacts/processed_feature_names.txt"
            )

            logged_flag = True

        # -------- FOLD RUN --------
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):

            mlflow.sklearn.log_model(model, name="model")

            # predictions
            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict_proba(X_test)
            final_test_preds += test_preds / config.N_FOLDS

            # feature importance
            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=3,
                n_jobs=1,
                random_state=config.SEED
            )

            feature_importance.append(fi.importances_mean)

            del model, oof_preds, test_preds, fi, X_train, X_test
            gc.collect()

    # -------- FINAL METRICS --------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)

    metrics = {
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": np.std(bacc_scores),
        "std_logloss": np.std(ll_scores),
    }

    recall_per_class = compute_recall_per_class(y, final_oof_preds)
    for cls, rec in zip(classes, recall_per_class):
        metrics[f"recall_class_{cls}"] = rec

    mlflow.log_metrics(metrics)

    # -------- FEATURE IMPORTANCE --------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)

    fi_df = pd.DataFrame({
        "feature": X_valid.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")

    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)

# -------- SAVE FILES --------

oof_df = pd.DataFrame(final_oof_preds, columns=[str(c) for c in classes])
oof_df.insert(0, "id", train_ids)

save_csv_file(
    oof_df,
    os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv")
)

test_df = pd.DataFrame(final_test_preds, columns=[str(c) for c in classes])
test_df.insert(0, "id", test_ids)

save_csv_file(
    test_df,
    os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
)

# -------- OOF Residuals --------
y_onehot = np.eye(n_classes)[y]

oof_residuals = y_onehot - final_oof_preds

residual_df = pd.DataFrame(
    oof_residuals,
    columns=[f"residual_{c}" for c in classes]
)

residual_df.insert(0, "id", train_ids)

save_csv_file(
    residual_df,
    os.path.join(config.RESIDUAL_DIR, f"{RUN_NAME}_oof_residuals.csv")
)

2026/04/15 21:50:02 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 1 | Balanced Acc: 0.87699 | LogLoss: 0.26747


2026/04/15 21:51:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 2 | Balanced Acc: 0.87750 | LogLoss: 0.26633


2026/04/15 21:53:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 3 | Balanced Acc: 0.88030 | LogLoss: 0.26891


2026/04/15 21:54:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 4 | Balanced Acc: 0.87945 | LogLoss: 0.26744


2026/04/15 21:55:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 5 | Balanced Acc: 0.87851 | LogLoss: 0.26563
Saving file to: artifacts\oof\SGD_seed42_v10_oof_proba.csv
Saving file to: artifacts\test_proba\SGD_seed42_v10_test_proba.csv
Saving file to: artifacts\residuals\SGD_seed42_v10_oof_residuals.csv


### Catboost Native

In [ ]:
EXP_NAME = "CATBOOST"
RUN_NAME = f"CATBOOST_seed{config.SEED}_{RUN}"

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    # ---------------- PARAMS ----------------
    mlflow.log_params({
        "seed": config.SEED,
        "n_folds": config.N_FOLDS,
        "n_rows": len(X),
        "n_cols": X.shape[1],
        "n_cat_cols": len(cat_cols),
        "n_num_cols": len(num_cols),
        **config.CATBOOST_PARAMS
    })

    mlflow.log_text("\n".join(X.columns.tolist()), "artifacts/all_feature_names.txt")
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")

    dtype_map = {col: str(dtype) for col, dtype in X.dtypes.items()}
    mlflow.log_dict(dtype_map, "artifacts/feature_dtypes.json")

    # ---------------- CV SETUP ----------------
    skf = StratifiedKFold(
        n_splits=config.N_FOLDS,
        shuffle=True,
        random_state=config.SEED
    )

    test_pool = cb.Pool(test_data, cat_features=cat_cols)
    n_classes = len(np.unique(y))

    final_oof_preds = np.zeros((len(X), n_classes))
    final_test_preds = np.zeros((len(test_data), n_classes))
    feature_importance = []

    bacc_scores = []
    ll_scores = []

    # ---------------- TRAINING ----------------
    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(skf.split(X, y), start=1),
        colour='green',
        ncols=300,
        total=config.N_FOLDS,
        desc='Model Training'
    ):

        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):

            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

            train_pool = cb.Pool(X_train, y_train, cat_features=cat_cols)
            valid_pool = cb.Pool(X_valid, y_valid, cat_features=cat_cols)

            model = cb.train(
                train_pool,
                config.CATBOOST_PARAMS,
                eval_set=valid_pool,
                verbose=False
            )

            # Log model
            mlflow.catboost.log_model(model, name='model')

            # predictions
            oof_preds = model.predict(valid_pool, prediction_type="Probability")
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict(test_pool, prediction_type="Probability")
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=3,
                n_jobs=-1,
                random_state=config.SEED
            )

            feature_importance.append(fi.importances_mean)
            
			# ---------------- CLEANUP ----------------
            del model, train_pool, valid_pool, oof_preds, test_preds, fi
            gc.collect()

    # ---------------- FINAL METRICS ----------------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    bacc_std = np.std(bacc_scores)
    ll_std = np.std(ll_scores)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    mlflow.log_metrics({
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": bacc_std,
        "std_logloss": ll_std,
        "recall_class_0": recall_per_class[0],
        "recall_class_1": recall_per_class[1],
        "recall_class_2": recall_per_class[2],
    })

    # ---------------- FEATURE IMPORTANCE ----------------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)
    fi_df = pd.DataFrame({
        "feature": X.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")

    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)

# ---------- Saving Files ----------

# OOF probabilities
oof_df = pd.DataFrame(
    final_oof_preds,
    columns=["0", "1", "2"]
)
oof_df.insert(0, "id", train_ids)

save_csv_file(
    oof_df,
    os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv")
)

# Test probabilities
test_df = pd.DataFrame(
    final_test_preds,
    columns=["0", "1", "2"]
)
test_df.insert(0, "id", test_ids)

save_csv_file(
    test_df,
    os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
)

Model Training:   0%|                                                                                         …

Fold 1 | Balanced Acc: 0.96232 | LogLoss: 0.19326
Fold 2 | Balanced Acc: 0.96596 | LogLoss: 0.09043
Fold 3 | Balanced Acc: 0.96625 | LogLoss: 0.33811
Fold 4 | Balanced Acc: 0.96282 | LogLoss: 0.26360
Fold 5 | Balanced Acc: 0.96333 | LogLoss: 0.25823
Saving file to: artifacts\oof\CATBOOST_seed42_v3_oof_proba.csv
Saving file to: artifacts\test_proba\CATBOOST_seed42_v3_test_proba.csv


### XGBoost TD

In [408]:
xgb_keep = [
    "soil_moisture",
    "crop_growth_stage",
    "temperature_c",
    "mulching_used",
    "wind_speed_kmh",
    "rainfall_mm",
]

X = X[xgb_keep]
test_data = test_data[xgb_keep]

In [409]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [411]:
EXP_NAME = "XGBoost_TD"
RUN_NAME = f"Xgbm_TD_seed{config.SEED}_{RUN}"

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.log_params({
        "seed": config.SEED,
        "n_folds": config.N_FOLDS,
        "n_rows": len(X),
        "n_cols": X.shape[1],
        "n_cat_cols": len(cat_cols),
        "n_num_cols": len(num_cols),
        **config.XGB_TD_PARAMS
    })

    mlflow.log_text("\n".join(X.columns.tolist()), "artifacts/all_feature_names.txt")
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")

    dtype_map = {col: str(dtype) for col, dtype in X.dtypes.items()}
    mlflow.log_dict(dtype_map, "artifacts/feature_dtypes.json")

    skf = StratifiedKFold(
        n_splits=config.N_FOLDS,
        shuffle=True,
        random_state=config.SEED
    )

    classes = np.unique(y)

    final_oof_preds = np.zeros((len(X), n_classes))
    final_test_preds = np.zeros((len(test_data), n_classes))
    feature_importance = []

    bacc_scores = []
    ll_scores = []

    logged_flag = False

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(skf.split(X, y), start=1),
        colour='green',
        ncols=300,
        total=config.N_FOLDS,
        desc='Model Training'
    ):

        X_train = X.iloc[train_idx].copy()
        X_valid = X.iloc[valid_idx].copy()
        X_test = test_data.copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        fold_cat_cols = cat_cols

        # -------- LOG ONCE --------
        if not logged_flag:
            mlflow.log_params({
                "seed": config.SEED,
                "n_folds": config.N_FOLDS,
                "n_rows": len(X),
                "n_cols_original": X.shape[1],
                "n_cols_processed": X_train.shape[1],
                **config.XGB_TD_PARAMS
            })

            mlflow.log_text(
                "\n".join(X_train.columns.astype(str).tolist()),
                "artifacts/processed_feature_names.txt"
            )

            logged_flag = True

        # -------- FOLD RUN --------
        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):

            model = XGB_TD_Classifier(**config.XGB_TD_PARAMS)

            model.fit(X_train, y_train, X_valid, y_valid, cat_col_names=fold_cat_cols)

            mlflow.sklearn.log_model(model, name="model")

            # predictions
            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")

            test_preds = model.predict_proba(X_test)
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=3,
                n_jobs=1,
                random_state=config.SEED
            )

            feature_importance.append(fi.importances_mean)

            del model, oof_preds, test_preds, fi, X_train, X_test
            gc.collect()

    # ---------------- FINAL METRICS ----------------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    bacc_std = np.std(bacc_scores)
    ll_std = np.std(ll_scores)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    mlflow.log_metrics({
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": bacc_std,
        "std_logloss": ll_std,
        "recall_class_0": recall_per_class[0],
        "recall_class_1": recall_per_class[1],
        "recall_class_2": recall_per_class[2]
    })

    # ---------------- FEATURE IMPORTANCE ----------------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)
    fi_df = pd.DataFrame({
        "feature": X_valid.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")

    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)


# ---------- Saving Files ----------

oof_df = pd.DataFrame(final_oof_preds, columns=[str(c) for c in classes])
oof_df.insert(0, "id", train_ids)

save_csv_file(
    oof_df,
    os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv")
)

test_df = pd.DataFrame(final_test_preds, columns=[str(c) for c in classes])
test_df.insert(0, "id", test_ids)

save_csv_file(
    test_df,
    os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
)

# -------- OOF Residuals --------
y_onehot = np.eye(n_classes)[y]

oof_residuals = y_onehot - final_oof_preds

residual_df = pd.DataFrame(
    oof_residuals,
    columns=[f"residual_{c}" for c in classes]
)

residual_df.insert(0, "id", train_ids)

save_csv_file(
    residual_df,
    os.path.join(config.RESIDUAL_DIR, f"{RUN_NAME}_oof_residuals.csv")
)

Model Training:   0%|                                                                                         …

2026/04/15 23:46:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/15 23:46:36 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Fold 1 | Balanced Acc: 0.96193 | LogLoss: 0.06082


2026/04/15 23:46:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/15 23:47:13 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Fold 2 | Balanced Acc: 0.96286 | LogLoss: 0.06126


...

Fold 3 | Balanced Acc: 0.96215 | LogLoss: 0.06149


2026/04/15 23:48:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/15 23:48:31 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Fold 4 | Balanced Acc: 0.96116 | LogLoss: 0.06210


2026/04/15 23:48:48 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/04/15 23:48:57 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.21.0+cu126) contains a local version label (+cu126). MLflow logged a pip requirement for this package as 'torchvision==0.21.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


Fold 5 | Balanced Acc: 0.96199 | LogLoss: 0.06102
Saving file to: artifacts\oof\Xgbm_TD_seed42_v3_oof_proba.csv
Saving file to: artifacts\test_proba\Xgbm_TD_seed42_v3_test_proba.csv
Saving file to: artifacts\residuals\Xgbm_TD_seed42_v3_oof_residuals.csv


### HistGBM

In [320]:
# List of numerical and categorical features
num_cols = X.select_dtypes(include='number').columns.to_list()
cat_cols = X.select_dtypes(exclude='number').columns.to_list()

In [321]:
EXP_NAME = "HISTGBM"
RUN_NAME = f"HGB_seed{config.SEED}_{RUN}"

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.log_params({
        "seed": config.SEED,
        "n_folds": config.N_FOLDS,
        "n_rows": len(X),
        "n_cols": X.shape[1],
        "n_cat_cols": len(cat_cols),
        "n_num_cols": len(num_cols),
        **config.HISTGBM_PARAMS
    })

    mlflow.log_text("\n".join(X.columns.tolist()), "artifacts/all_feature_names.txt")
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")

    dtype_map = {col: str(dtype) for col, dtype in X.dtypes.items()}
    mlflow.log_dict(dtype_map, "artifacts/feature_dtypes.json")

    skf = StratifiedKFold(
        n_splits=config.N_FOLDS,
        shuffle=True,
        random_state=config.SEED
    )

    n_classes = len(np.unique(y))

    final_oof_preds = np.zeros((len(X), n_classes))
    final_test_preds = np.zeros((len(test_data), n_classes))
    feature_importance = []

    bacc_scores = []
    ll_scores = []

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(skf.split(X, y), start=1),
        colour='green',
        ncols=300,
        total=config.N_FOLDS,
        desc='Model Training'
    ):

        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):

            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
            
            X_train[cat_cols] = X_train[cat_cols].astype('category')
            X_valid[cat_cols] = X_valid[cat_cols].astype('category')

            model = HistGradientBoostingClassifier(**config.HISTGBM_PARAMS)

            model.fit(X_train, y_train)

            mlflow.sklearn.log_model(model, name="model")

            # predictions
            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")
            
            test_data[cat_cols] = test_data[cat_cols].astype('category')
            test_preds = model.predict_proba(test_data)
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=3,
                n_jobs=-1,
                random_state=config.SEED
            )

            feature_importance.append(fi.importances_mean)
            
			# ---------------- CLEANUP ----------------
            del model, oof_preds, test_preds, fi
            gc.collect()

    # ---------------- FINAL METRICS ----------------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    bacc_std = np.std(bacc_scores)
    ll_std = np.std(ll_scores)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    mlflow.log_metrics({
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": bacc_std,
        "std_logloss": ll_std,
        "recall_class_0": recall_per_class[0],
        "recall_class_1": recall_per_class[1],
        "recall_class_2": recall_per_class[2],
    })

    # ---------------- FEATURE IMPORTANCE ----------------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)
    fi_df = pd.DataFrame({
        "feature": X.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")

    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)


# ---------- Saving Files ----------

# OOF probabilities
oof_df = pd.DataFrame(
    final_oof_preds,
    columns=["0", "1", "2"]
)
oof_df.insert(0, "id", train_ids)

save_csv_file(
    oof_df,
    os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv")
)

# Test probabilities
test_df = pd.DataFrame(
    final_test_preds,
    columns=["0", "1", "2"]
)
test_df.insert(0, "id", test_ids)

save_csv_file(
    test_df,
    os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
)

# -------- OOF Residuals --------
y_onehot = np.eye(n_classes)[y]

oof_residuals = y_onehot - final_oof_preds

residual_df = pd.DataFrame(
    oof_residuals,
    columns=[f"residual_{c}" for c in classes]
)

residual_df.insert(0, "id", train_ids)

save_csv_file(
    residual_df,
    os.path.join(config.RESIDUAL_DIR, f"{RUN_NAME}_oof_residuals.csv")
)

Model Training:   0%|                                                                                         …

2026/04/15 22:34:39 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Fold 1 | Balanced Acc: 0.96218 | LogLoss: 0.05662


KeyboardInterrupt: 

### Resnet_RTDL_D

In [ ]:
EXP_NAME = "RESNET_RTDL_D"
RUN_NAME = f"Resnet_seed{config.SEED}_{RUN}"

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.log_params({
        "seed": config.SEED,
        "n_folds": config.N_FOLDS,
        "n_rows": len(X),
        "n_cols": X.shape[1],
        "n_cat_cols": len(cat_cols),
        "n_num_cols": len(num_cols),
        **config.Resnet_RTDL_D_PARAMS
    })

    mlflow.log_text("\n".join(X.columns.tolist()), "artifacts/all_feature_names.txt")
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")

    dtype_map = {col: str(dtype) for col, dtype in X.dtypes.items()}
    mlflow.log_dict(dtype_map, "artifacts/feature_dtypes.json")

    skf = StratifiedKFold(
        n_splits=config.N_FOLDS,
        shuffle=True,
        random_state=config.SEED
    )

    n_classes = len(np.unique(y))

    final_oof_preds = np.zeros((len(X), n_classes))
    final_test_preds = np.zeros((len(test_data), n_classes))
    feature_importance = []

    bacc_scores = []
    ll_scores = []

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(skf.split(X, y), start=1),
        colour='green',
        ncols=300,
        total=config.N_FOLDS,
        desc='Model Training'
    ):

        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):

            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
            
            X_train[cat_cols] = X_train[cat_cols].astype('category')
            X_valid[cat_cols] = X_valid[cat_cols].astype('category')

            model = Resnet_RTDL_D_Classifier(**config.Resnet_RTDL_D_PARAMS)

            model.fit(X_train, y_train, X_valid, y_valid, cat_col_names=cat_cols)

            mlflow.sklearn.log_model(model, name="model")

            # predictions
            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")
            
            test_data[cat_cols] = test_data[cat_cols].astype('category')
            test_preds = model.predict_proba(test_data)
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=3,
                n_jobs=-1,
                random_state=config.SEED
            )

            feature_importance.append(fi.importances_mean)
            
			# ---------------- CLEANUP ----------------
            del model, oof_preds, test_preds, fi
            gc.collect()

    # ---------------- FINAL METRICS ----------------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    bacc_std = np.std(bacc_scores)
    ll_std = np.std(ll_scores)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    mlflow.log_metrics({
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": bacc_std,
        "std_logloss": ll_std,
        "recall_class_0": recall_per_class[0],
        "recall_class_1": recall_per_class[1],
        "recall_class_2": recall_per_class[2],
    })

    # ---------------- FEATURE IMPORTANCE ----------------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)
    fi_df = pd.DataFrame({
        "feature": X.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")

    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)


# ---------- Saving Files ----------

# OOF probabilities
oof_df = pd.DataFrame(
    final_oof_preds,
    columns=["0", "1", "2"]
)
oof_df.insert(0, "id", train_ids)

save_csv_file(
    oof_df,
    os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv")
)

# Test probabilities
test_df = pd.DataFrame(
    final_test_preds,
    columns=["0", "1", "2"]
)
test_df.insert(0, "id", test_ids)

save_csv_file(
    test_df,
    os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
)

### TabM_D

In [ ]:
EXP_NAME = "TabM_D"
RUN_NAME = f"TabM_seed{config.SEED}_{RUN}"

mlflow.set_experiment(EXP_NAME)

with mlflow.start_run(run_name=RUN_NAME):

    mlflow.log_params({
        "seed": config.SEED,
        "n_folds": config.N_FOLDS,
        "n_rows": len(X),
        "n_cols": X.shape[1],
        "n_cat_cols": len(cat_cols),
        "n_num_cols": len(num_cols),
        **config.TabM_D_PARAMS
    })

    mlflow.log_text("\n".join(X.columns.tolist()), "artifacts/all_feature_names.txt")
    mlflow.log_text("\n".join(cat_cols), "artifacts/cat_cols.txt")
    mlflow.log_text("\n".join(num_cols), "artifacts/num_cols.txt")

    dtype_map = {col: str(dtype) for col, dtype in X.dtypes.items()}
    mlflow.log_dict(dtype_map, "artifacts/feature_dtypes.json")

    skf = StratifiedKFold(
        n_splits=config.N_FOLDS,
        shuffle=True,
        random_state=config.SEED
    )

    n_classes = len(np.unique(y))

    final_oof_preds = np.zeros((len(X), n_classes))
    final_test_preds = np.zeros((len(test_data), n_classes))
    feature_importance = []

    bacc_scores = []
    ll_scores = []

    for fold, (train_idx, valid_idx) in tqdm(
        enumerate(skf.split(X, y), start=1),
        colour='green',
        ncols=300,
        total=config.N_FOLDS,
        desc='Model Training'
    ):

        with mlflow.start_run(run_name=f"fold_{fold}", nested=True):

            X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
            y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
            
            X_train[cat_cols] = X_train[cat_cols].astype('category')
            X_valid[cat_cols] = X_valid[cat_cols].astype('category')

            model = TabM_D_Classifier(**config.TabM_D_PARAMS)

            model.fit(X_train, y_train, X_valid, y_valid, cat_col_names=cat_cols)

            mlflow.sklearn.log_model(model, name="model")

            # predictions
            oof_preds = model.predict_proba(X_valid)
            final_oof_preds[valid_idx] = oof_preds

            bacc = compute_bacc(y_valid, oof_preds)
            ll = compute_logloss(y_valid, oof_preds)

            bacc_scores.append(bacc)
            ll_scores.append(ll)

            print(f"Fold {fold} | Balanced Acc: {bacc:.5f} | LogLoss: {ll:.5f}")
            
            test_data[cat_cols] = test_data[cat_cols].astype('category')
            test_preds = model.predict_proba(test_data)
            final_test_preds += test_preds / config.N_FOLDS

            fi = permutation_importance(
                model,
                X_valid,
                y_valid,
                scoring=agnostic_bacc_scorer,
                n_repeats=3,
                n_jobs=-1,
                random_state=config.SEED
            )

            feature_importance.append(fi.importances_mean)
            
			# ---------------- CLEANUP ----------------
            del model, oof_preds, test_preds, fi
            gc.collect()

    # ---------------- FINAL METRICS ----------------
    oof_bacc = compute_bacc(y, final_oof_preds)
    oof_logloss = compute_logloss(y, final_oof_preds)
    bacc_std = np.std(bacc_scores)
    ll_std = np.std(ll_scores)
    recall_per_class = compute_recall_per_class(y, final_oof_preds)

    mlflow.log_metrics({
        "oof_bacc": oof_bacc,
        "oof_logloss": oof_logloss,
        "std_bacc": bacc_std,
        "std_logloss": ll_std,
        "recall_class_0": recall_per_class[0],
        "recall_class_1": recall_per_class[1],
        "recall_class_2": recall_per_class[2],
    })

    # ---------------- FEATURE IMPORTANCE ----------------
    fi_mean = np.mean(np.vstack(feature_importance), axis=0)
    fi_df = pd.DataFrame({
        "feature": X.columns,
        "importance": fi_mean
    }).sort_values("importance", ascending=False).head(50)

    fig, ax = plt.subplots(figsize=(10, 14))
    ax.barh(fi_df["feature"][::-1], fi_df["importance"][::-1])
    ax.set_title("Top 50 Feature Importance")

    fig.tight_layout()
    mlflow.log_figure(fig, "feature_importance/fi.png")
    plt.close(fig)


# ---------- Saving Files ----------

# OOF probabilities
oof_df = pd.DataFrame(
    final_oof_preds,
    columns=["0", "1", "2"]
)
oof_df.insert(0, "id", train_ids)

save_csv_file(
    oof_df,
    os.path.join(config.OOF_DIR, f"{RUN_NAME}_oof_proba.csv")
)

# Test probabilities
test_df = pd.DataFrame(
    final_test_preds,
    columns=["0", "1", "2"]
)
test_df.insert(0, "id", test_ids)

save_csv_file(
    test_df,
    os.path.join(config.TEST_PROBA_DIR, f"{RUN_NAME}_test_proba.csv")
)